In [ ]:
#!/usr/bin/env python3
"""
   ResNet-18 End-to-End Training: Dark Matter Substructure Classification    
                                                                              
   KEY LESSON: 3-stage freeze strategy is WRONG for ResNet-18.               
   - ConvNeXt V2 (196M params) needs staged unfreezing to protect pretrained  
     features from being overwritten by cold-start head gradients.            
   - ResNet-18 (11.2M params) with a replaced conv1 needs co-adaptation:     
     conv1, backbone, and head must all learn together from epoch 1.          
   - This is exactly what the original notebook (0.9743 AUC) does.            
                                                                              
   Strategy: single training loop, all params unfrozen from epoch 1,         
              OneCycleLR (same as original notebook), proper augmentation,    
              larger batch, EMA, more epochs.                                 
                                                                              
   conv1 init: pretrained RGB weights summed → 1-channel (not random)        
   Output dir: /kaggle/working/resnet18_e2e/                                  

"""

import os, math, random, gc, warnings
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR

import timm
from timm.utils import ModelEma

from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split

import albumentations as A
from albumentations.pytorch import ToTensorV2

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
warnings.filterwarnings('ignore')


# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════
class CFG:
    DATA_ROOT  = "/kaggle/input/datasets/stellarquant/deeplensetask1/dataset"
    OUTPUT_DIR = "/kaggle/working/resnet18_e2e"

    MODEL_NAME  = "resnet18"
    NUM_CLASSES = 3
    IMG_SIZE    = 224
    DROP_RATE   = 0.1

    # Single flat training loop — no staging
    # Original notebook: 15 epochs, batch 32 → 30k/32 = 938 steps/ep
    # Ours: 60 epochs, batch 256 → 27k/256 = 106 steps/ep
    # More epochs compensates for fewer steps per epoch at larger batch
    EPOCHS      = 60
    PATIENCE    = 15        # early stopping on val AUC

    BATCH_SIZE  = 256
    NUM_WORKERS = 4
    VAL_SPLIT   = 0.10

    # OneCycleLR — same scheduler family as original notebook
    MAX_LR      = 1e-3      # same as original notebook's max_lr
    PCT_START   = 0.1       # 10% warmup — same as original
    DIV_FACTOR  = 25        # initial lr = max_lr / div_factor = 4e-5
    FINAL_DIV   = 1e4       # final lr = max_lr / final_div = 1e-7

    WEIGHT_DECAY    = 0.01
    LABEL_SMOOTHING = 0.1
    MIXUP_ALPHA     = 0.4
    EMA_DECAY       = 0.9999
    GRAD_CLIP       = 1.0
    USE_BF16        = True

    CLASS_NAMES = ['no_sub', 'subhalo', 'vortex']
    CLASS_DIRS  = {'no_sub': 'no', 'subhalo': 'sphere', 'vortex': 'vort'}
    PIXEL_MEAN  = 0.0615
    PIXEL_STD   = 0.1152
    SEED        = 42



# REPRODUCIBILITY
def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


# EARLY STOPPING
class EarlyStopping:
    def __init__(self, patience):
        self.patience  = patience
        self.best      = -np.inf
        self.counter   = 0
        self.triggered = False

    def step(self, metric):
        if metric > self.best:
            self.best = metric; self.counter = 0; return True
        self.counter += 1
        if self.counter >= self.patience:
            self.triggered = True
        return False

    @property
    def status(self):
        return f'  [ES {self.counter}/{self.patience}]' if self.counter else ''


# DATASET
class LensDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = file_paths
        self.labels     = labels
        self.transform  = transform

    def __len__(self): return len(self.file_paths)

    def __getitem__(self, idx):
        img = np.load(self.file_paths[idx]).astype(np.float32)
        if img.ndim == 3: img = img[0]
        img_u8 = (img * 255).clip(0, 255).astype(np.uint8)
        if self.transform:
            return self.transform(image=img_u8)['image'], self.labels[idx]
        return torch.from_numpy(img_u8[None]).float() / 255.0, self.labels[idx]


def build_file_list(root, split):
    paths, labels = [], []
    for i, cls in enumerate(CFG.CLASS_NAMES):
        cls_dir   = root / split / CFG.CLASS_DIRS[cls]
        npy_files = sorted(cls_dir.glob("*.npy")) + sorted(cls_dir.glob("*.NPY"))
        if not npy_files:
            raise FileNotFoundError(f"No .npy files in {cls_dir}")
        paths.extend(str(p) for p in npy_files)
        labels.extend([i] * len(npy_files))
        print(f"  [{split}/{cls}]  {len(npy_files):,} images")
    return paths, labels


# AUGMENTATIONS
def get_train_transforms():
    return A.Compose([
        A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE, interpolation=2),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Rotate(limit=180, p=0.9, border_mode=0, value=0),
        A.RandomResizedCrop(
            size=(CFG.IMG_SIZE, CFG.IMG_SIZE),
            scale=(0.90, 1.00), ratio=(0.95, 1.05),
            interpolation=2, p=0.5,
        ),
        A.GaussNoise(var_limit=(0.65, 2.60), p=0.35),
        A.CoarseDropout(
            max_holes=4, max_height=18, max_width=18,
            min_holes=1, min_height=8,  min_width=8,
            fill_value=0, p=0.20,
        ),
        A.Normalize(mean=[CFG.PIXEL_MEAN], std=[CFG.PIXEL_STD],
                    max_pixel_value=255.0),
        ToTensorV2(),
    ])


def get_val_transforms():
    return A.Compose([
        A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE, interpolation=2),
        A.Normalize(mean=[CFG.PIXEL_MEAN], std=[CFG.PIXEL_STD],
                    max_pixel_value=255.0),
        ToTensorV2(),
    ])


# MIXUP
def mixup_data(x, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def mixup_loss(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


# MODEL
# conv1 initialised from pretrained weights (sum of RGB channels → 1 channel).
# All parameters trained end-to-end from epoch 1 — no staged freezing.

class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.resnet = timm.create_model(
            CFG.MODEL_NAME, pretrained=True, drop_rate=CFG.DROP_RATE
        )
        # Save pretrained conv1 weights before replacing the layer
        pretrained_w = self.resnet.conv1.weight.data          # (64, 3, 7, 7)

        # Replace conv1: 3-channel → 1-channel
        self.resnet.conv1 = nn.Conv2d(
            1, 64, kernel_size=7, stride=2, padding=3, bias=False
        )
        # Initialise from pretrained: sum RGB channels → (64, 1, 7, 7)
        # Preserves learned edge/texture detectors, just collapsed to 1 channel
        self.resnet.conv1.weight.data = pretrained_w.sum(dim=1, keepdim=True)

        # Replace head
        self.resnet.fc = nn.Linear(512, CFG.NUM_CLASSES, bias=True)

    def forward(self, x):
        return self.resnet(x)


def build_model():
    model = SimpleModel()
    total  = sum(p.numel() for p in model.parameters())
    head   = sum(p.numel() for p in model.resnet.fc.parameters())
    print(f"  Model  : ResNet-18  |  {total/1e6:.1f}M params total")
    print(f"  conv1  : 1→64ch, init = sum of pretrained RGB filters")
    print(f"  Head   : Linear(512→3), {head/1e3:.1f}K params")
    print(f"  Freeze : NONE — full end-to-end training from epoch 1")
    return model


# TRAIN / EVAL

def train_one_epoch(model, loader, optimizer, scheduler, criterion, ema, device, ep):
    model.train()
    dtype  = torch.bfloat16 if CFG.USE_BF16 else torch.float32
    losses = []
    pbar   = tqdm(loader, desc=f'Ep {ep:02d}', leave=True)
    for imgs, labels in pbar:
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        imgs, y_a, y_b, lam = mixup_data(imgs, labels, CFG.MIXUP_ALPHA)
        with torch.autocast('cuda', dtype=dtype):
            logits = model(imgs)
            loss   = mixup_loss(criterion, logits, y_a, y_b, lam)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
        optimizer.step()
        scheduler.step()          # OneCycleLR steps every batch
        ema.update(model)
        losses.append(loss.item())
        pbar.set_postfix({'loss': f'{loss.item():.4f}',
                          'lr':   f'{scheduler.get_last_lr()[0]:.2e}'})
    return float(np.mean(losses))


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    dtype = torch.bfloat16 if CFG.USE_BF16 else torch.float32
    all_probs, all_labels = [], []
    for imgs, labels in tqdm(loader, desc='Eval', leave=False):
        imgs = imgs.to(device, non_blocking=True)
        with torch.autocast('cuda', dtype=dtype):
            probs = F.softmax(model(imgs), dim=-1)
        all_probs.append(probs.cpu().float().numpy())
        all_labels.append(labels.numpy())
    probs  = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)
    macro  = roc_auc_score(labels, probs, multi_class='ovr', average='macro')
    per_cls = {cls: roc_auc_score((labels==i).astype(int), probs[:,i])
               for i, cls in enumerate(CFG.CLASS_NAMES)}
    return macro, per_cls, probs, labels


# PLOTS
def plot_roc(labels, probs, save_path, title=""):
    COLORS = ['#e74c3c', '#2ecc71', '#3498db']
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(f'ROC Curves — {title}', fontsize=13, fontweight='bold')
    ax, aucs = axes[0], []
    for i, (cls, col) in enumerate(zip(CFG.CLASS_NAMES, COLORS)):
        fpr, tpr, _ = roc_curve((labels==i).astype(int), probs[:,i])
        auc_val = roc_auc_score((labels==i).astype(int), probs[:,i])
        aucs.append(auc_val)
        ax.plot(fpr, tpr, color=col, lw=2.2,
                label=f'{cls}  (AUC={auc_val:.4f})')
    ax.plot([0,1],[0,1],'k--',lw=1); ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title('One-vs-Rest ROC'); ax.legend(loc='lower right'); ax.grid(alpha=0.25)
    macro = np.mean(aucs)
    ax2   = axes[1]
    bars  = ax2.bar(CFG.CLASS_NAMES, aucs, color=COLORS, alpha=0.85,
                    edgecolor='white', linewidth=1.2)
    ax2.axhline(macro, color='gold', lw=2, ls='--', label=f'Macro={macro:.4f}')
    ax2.set_ylim(max(0.5, min(aucs)-0.03), 1.005)
    ax2.set_ylabel('AUC'); ax2.set_title('Per-Class AUC'); ax2.legend()
    ax2.grid(axis='y', alpha=0.25)
    for bar, v in zip(bars, aucs):
        ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
                 f'{v:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight'); plt.close()
    print(f"  ROC plot → {save_path}")


def plot_history(history, save_path):
    eps = range(1, len(history['loss'])+1)
    fig, ax1 = plt.subplots(figsize=(13, 5))
    ax1.plot(eps, history['loss'], color='#e74c3c', lw=2, label='Train Loss')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss', color='#e74c3c')
    ax2 = ax1.twinx()
    ax2.plot(eps, history['val_auc'], color='#2ecc71', lw=2.2,
             label='Val Macro AUC (EMA)')
    ax2.set_ylabel('Macro AUC', color='#2ecc71')
    lines  = ax1.get_legend_handles_labels()
    lines2 = ax2.get_legend_handles_labels()
    ax1.legend(lines[0]+lines2[0], lines[1]+lines2[1], loc='center right')
    ax1.set_title(f'ResNet-18 End-to-End  '
                  f'(best val AUC={max(history["val_auc"]):.4f})', fontsize=12)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight'); plt.close()
    print(f"  History  → {save_path}")


# MAIN
def main():
    seed_everything(CFG.SEED)
    os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"  GPU  : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB\n")

    # File lists 
    print(" Loading file lists ")
    root = Path(CFG.DATA_ROOT)
    train_paths, train_labels = build_file_list(root, 'train')
    test_paths,  test_labels  = build_file_list(root, 'val')

    tr_p, val_p, tr_l, val_l = train_test_split(
        train_paths, train_labels,
        test_size=CFG.VAL_SPLIT, stratify=train_labels, random_state=CFG.SEED,
    )
    print(f"\n  Train={len(tr_p):,}  Val={len(val_p):,}  Test={len(test_paths):,}\n")

    train_loader = DataLoader(
        LensDataset(tr_p,  tr_l,  get_train_transforms()),
        batch_size=CFG.BATCH_SIZE, shuffle=True,
        num_workers=CFG.NUM_WORKERS, pin_memory=True,
        drop_last=True, persistent_workers=True,
    )
    val_loader = DataLoader(
        LensDataset(val_p, val_l, get_val_transforms()),
        batch_size=CFG.BATCH_SIZE*2, shuffle=False,
        num_workers=CFG.NUM_WORKERS, pin_memory=True, persistent_workers=True,
    )
    test_loader = DataLoader(
        LensDataset(test_paths, test_labels, get_val_transforms()),
        batch_size=CFG.BATCH_SIZE*2, shuffle=False,
        num_workers=CFG.NUM_WORKERS, pin_memory=True, persistent_workers=True,
    )
    steps_per_epoch = len(train_loader)
    total_steps     = CFG.EPOCHS * steps_per_epoch
    print(f"  Steps/epoch={steps_per_epoch}  |  Total steps={total_steps}\n")

    # Model, optimiser, scheduler
    print("── Building model ──")
    model     = build_model().to(device)
    ema       = ModelEma(model, decay=CFG.EMA_DECAY, device=device)
    criterion = nn.CrossEntropyLoss(label_smoothing=CFG.LABEL_SMOOTHING)
    ckpt_path = f"{CFG.OUTPUT_DIR}/best_model.pth"

    optimizer = AdamW(model.parameters(), lr=CFG.MAX_LR,
                      weight_decay=CFG.WEIGHT_DECAY)

    # OneCycleLR — same scheduler as original notebook
    scheduler = OneCycleLR(
        optimizer,
        max_lr       = CFG.MAX_LR,
        total_steps  = total_steps,
        pct_start    = CFG.PCT_START,
        anneal_strategy = 'cos',
        div_factor   = CFG.DIV_FACTOR,
        final_div_factor = CFG.FINAL_DIV,
    )

    # Training loop
    print(f"\n{'='*62}")
    print(f"  TRAINING — end-to-end, all params, {CFG.EPOCHS} epochs max")
    print(f"{'='*62}\n")

    history      = {'loss': [], 'val_auc': []}
    best_val_auc = 0.0
    es           = EarlyStopping(patience=CFG.PATIENCE)

    for ep in range(1, CFG.EPOCHS + 1):
        loss = train_one_epoch(model, train_loader, optimizer, scheduler,
                               criterion, ema, device, ep)
        macro, per_cls, _, _ = evaluate(ema.ema, val_loader, device)
        history['loss'].append(loss)
        history['val_auc'].append(macro)

        cls_str = '  '.join(f'{k}={v:.4f}' for k, v in per_cls.items())
        improved = es.step(macro)
        print(f"  [Ep {ep:02d}/{CFG.EPOCHS}]  loss={loss:.4f}  "
              f"macro_auc={macro:.4f}  |  {cls_str}{es.status}")

        if improved:
            best_val_auc = macro
            torch.save({'epoch': ep, 'model': ema.ema.state_dict(),
                        'val_auc': macro}, ckpt_path)
            print(f"  ✓  New best val AUC={macro:.4f}  → checkpoint saved")

        if es.triggered:
            print(f"\n  ⏹  Early stop at ep={ep}  "
                  f"best={es.best:.4f}  (no gain for {CFG.PATIENCE} ep)")
            break

    # Final evaluation
    print(f"\n{'='*62}")
    print("  FINAL EVALUATION  (best EMA checkpoint)")
    print(f"{'='*62}")

    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model'])
    print(f"  Loaded checkpoint from epoch {ckpt['epoch']}  "
          f"(val AUC={ckpt['val_auc']:.4f})")

    test_auc, test_per_cls, test_probs, test_lbl = evaluate(
        model, test_loader, device
    )

    plot_roc(test_lbl, test_probs,
             save_path=f"{CFG.OUTPUT_DIR}/roc_curves.png",
             title=f"ResNet-18 E2E — Test AUC={test_auc:.4f}")
    plot_history(history, save_path=f"{CFG.OUTPUT_DIR}/training_history.png")

    print(f"\n{'='*62}")
    print(f"  RESNET-18 END-TO-END — COMPLETE")
    print(f"{'='*62}")
    print(f"  Best val AUC : {best_val_auc:.4f}")
    print(f"  Test AUC     : {test_auc:.4f}")
    for cls, val in test_per_cls.items():
        print(f"    {cls:>10} : {val:.4f}")
    print(f"  Checkpoint   : {ckpt_path}")
    print(f"{'='*62}\n")


if __name__ == '__main__':
    main()

  GPU  : NVIDIA H100 80GB HBM3
  VRAM : 85.0 GB

── Loading file lists ──
  [train/no_sub]  10,000 images
  [train/subhalo]  10,000 images
  [train/vortex]  10,000 images
  [val/no_sub]  2,500 images
  [val/subhalo]  2,500 images
  [val/vortex]  2,500 images

  Train=27,000  Val=3,000  Test=7,500

  Steps/epoch=105  |  Total steps=6300

── Building model ──


model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

  Model  : ResNet-18  |  11.2M params total
  conv1  : 1→64ch, init = sum of pretrained RGB filters
  Head   : Linear(512→3), 1.5K params
  Freeze : NONE — full end-to-end training from epoch 1

  TRAINING — end-to-end, all params, 60 epochs max



Ep 01:   0%|          | 0/105 [00:00<?, ?it/s]

Eval:   0%|          | 0/6 [00:00<?, ?it/s]

...

Ep 60:   0%|          | 0/105 [00:00<?, ?it/s]

Eval:   0%|          | 0/6 [00:00<?, ?it/s]

  [Ep 60/60]  loss=0.8470  macro_auc=0.6506  |  no_sub=0.6936  subhalo=0.6039  vortex=0.6544
  ✓  New best val AUC=0.6506  → checkpoint saved

  FINAL EVALUATION  (best EMA checkpoint)
  Loaded checkpoint from epoch 60  (val AUC=0.6506)


Eval:   0%|          | 0/15 [00:00<?, ?it/s]

  ROC plot → /kaggle/working/resnet18_e2e/roc_curves.png
  History  → /kaggle/working/resnet18_e2e/training_history.png

  RESNET-18 END-TO-END — COMPLETE
  Best val AUC : 0.6506
  Test AUC     : 0.6420
        no_sub : 0.6840
       subhalo : 0.5872
        vortex : 0.6550
  Checkpoint   : /kaggle/working/resnet18_e2e/best_model.pth

